# CS 260D — Coreset fine-tuning after pruning (Colab launcher)

Run the cells top to bottom. **Runtime → Change runtime type → T4 GPU** first.

This notebook is generated by `scripts/build_notebook.py` and contains a copy of `src/`
(section 2). A matching branch checkout is required for Git provenance.
Do not edit code here: edit `src/` in the repo,
re-run `python scripts/build_notebook.py`, push, and re-open the notebook from GitHub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Clone the audit branch and verify embedded source provenance

In [ ]:
import pathlib, subprocess, sys
REPO = pathlib.Path('/content/cs260d-pruning-coreset')
BRANCH = 'fix/experiment-audit'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/liuyihao4fun/cs260d-pruning-coreset.git', str(REPO)], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip() == BRANCH
print('Git commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)

In [ ]:
EXPECTED_SOURCE_SHA256 = '591f58a3a95e65a50cd29a8537df74735ec817dcc31d66babe1e775d4ed3c3b8'

actual = subprocess.check_output([sys.executable, '-c', 'from src.audit import source_sha256; print(source_sha256())'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_SOURCE_SHA256, 'Notebook/source mismatch: open the notebook from the same branch revision.'
%cd /content/cs260d-pruning-coreset

## 1. Mount Google Drive (checkpoints + results survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/cs260d'        # shared folder for checkpoint + results
CKPT = f'{WORK}/resnet18_cifar10.pt'
DATA = f'{WORK}/data'                         # CIFAR cached on Drive: downloaded once, reused by later sessions
NAME = 'me'                                   # <-- use a unique identifier for each runtime
RESULTS = f'{WORK}/results_audit_{NAME}.jsonl'
import os; os.makedirs(WORK, exist_ok=True)

## 2. Write the code onto this machine

In [ ]:
import os
os.makedirs('/content/cs260d-pruning-coreset/src', exist_ok=True)
os.makedirs('/content/cs260d-pruning-coreset/src/selection', exist_ok=True)
%cd /content/cs260d-pruning-coreset

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/__init__.py

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/audit.py
"""Versioned experiment identity and append-only, single-writer result journals.

This module deliberately has no ML dependencies, so recovery can be checked on CPU.
Legacy records remain readable, but never satisfy a v2 resume check.
"""
import hashlib
import json
import math
import os
from pathlib import Path
import subprocess
import warnings
from datetime import datetime, timezone

SCHEMA_VERSION = 2
PROTOCOL = "experiment-audit-v2"
IDENTITY_FIELDS = (
    "dataset", "sparsity", "method", "frac", "epochs", "lr", "batch_size", "wd",
    "seed", "reselect_every", "no_weights", "damage_alpha", "limit_train", "tag",
)
TRAINING_FILES = (
    "audit.py", "data.py", "finetune.py", "models.py", "prune.py", "utils.py",
    "selection/__init__.py", "selection/random_sel.py", "selection/score_based.py",
    "selection/craig.py",
)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def source_sha256(root=None):
    root = Path(root) if root is not None else Path(__file__).resolve().parent
    digest = hashlib.sha256()
    for name in TRAINING_FILES:
        digest.update(name.encode() + b"\0" + (root / name).read_bytes() + b"\0")
    return digest.hexdigest()


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def identity_key(identity):
    return "v2_" + hashlib.sha256(canonical_json(identity).encode()).hexdigest()


def experiment_identity(a, checkpoint_sha256=None, source_hash=None):
    identity = {name: getattr(a, name) for name in IDENTITY_FIELDS}
    if a.method == "full":
        identity["frac"] = 1.0
    identity.update(
        protocol=PROTOCOL,
        checkpoint_sha256=checkpoint_sha256 or file_sha256(a.ckpt),
        source_sha256=source_hash or source_sha256(),
    )
    return identity


def run_key(a):
    return identity_key(experiment_identity(a))


def full_config(a):
    return {k: v for k, v in vars(a).items() if not k.startswith("_")}


def git_metadata():
    root = Path(__file__).resolve().parents[1]
    def git(*args):
        try:
            return subprocess.check_output(
                ["git", "-C", str(root), *args], stderr=subprocess.DEVNULL,
                text=True, timeout=10,
            ).strip()
        except (OSError, subprocess.SubprocessError):
            return None
    status = git("status", "--porcelain")
    return {"commit": git("rev-parse", "HEAD"),
            "branch": git("rev-parse", "--abbrev-ref", "HEAD"),
            "dirty": None if status is None else bool(status),
            "status_porcelain": status}


def is_completed(record):
    status = record.get("status")
    value = record.get("final_acc")
    completed = status == "completed" or (status is None and record.get("schema_version") != 2)
    return (completed and isinstance(value, (int, float))
            and not isinstance(value, bool) and math.isfinite(value))


def completed_keys(records):
    """Only explicit, internally consistent v2 completions can skip new work."""
    return {r["key"] for r in records
            if r.get("schema_version") == SCHEMA_VERSION
            and r.get("status") == "completed" and is_completed(r)
            and isinstance(r.get("identity"), dict)
            and r["identity"].get("protocol") == PROTOCOL
            and r.get("key") == identity_key(r["identity"])}


def _decode_records(raw, path):
    lines = raw.splitlines()
    records = []
    last = max((i for i, line in enumerate(lines) if line.strip()), default=-1)
    damaged_tail = None
    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except (ValueError, UnicodeDecodeError) as exc:
            # An earlier append may have preserved an interrupted fragment and
            # explicitly marked it. Never ignore arbitrary interior corruption.
            marker = None
            following = next((v for v in lines[i + 1:] if v.strip()), None)
            if following is not None:
                try:
                    marker = json.loads(following)
                except (ValueError, UnicodeDecodeError):
                    pass
            expected = hashlib.sha256(line).hexdigest()
            if (isinstance(marker, dict) and marker.get("status") == "journal_recovery"
                    and marker.get("damaged_line_sha256") == expected):
                warnings.warn(f"{path}:{i + 1}: preserved interrupted write", RuntimeWarning)
                continue
            if i == last:
                damaged_tail = line
                warnings.warn(f"{path}:{i + 1}: ignoring incomplete JSONL tail", RuntimeWarning)
                continue
            raise ValueError(f"{path}:{i + 1}: corrupt interior JSONL record") from exc
        if not isinstance(record, dict):
            raise ValueError(f"{path}:{i + 1}: JSONL record must be an object")
        records.append(record)
    return records, damaged_tail


def load_jsonl(path):
    p = Path(path)
    return _decode_records(p.read_bytes(), p)[0] if p.exists() else []


def append_jsonl(path, record):
    """Append + fsync without truncating bytes, even after a partial final write.

    Use one writer per file (including across Colab accounts). POSIX locks provide
    local process protection; Google Drive is not a distributed locking service.
    """
    import fcntl
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    payload = canonical_json(record).encode() + b"\n"
    with p.open("a+b") as f:
        fcntl.flock(f.fileno(), fcntl.LOCK_EX)
        try:
            f.seek(0)
            raw = f.read()
            _, tail = _decode_records(raw, p)
            if raw and not raw.endswith(b"\n"):
                f.write(b"\n")
            if tail is not None:
                marker = {"status": "journal_recovery", "timestamp": utc_now(),
                          "damaged_line_sha256": hashlib.sha256(tail).hexdigest()}
                f.write(canonical_json(marker).encode() + b"\n")
            f.write(payload)
            f.flush()
            os.fsync(f.fileno())
        finally:
            fcntl.flock(f.fileno(), fcntl.LOCK_UN)

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/data.py
"""CIFAR data held entirely on the GPU, with batched GPU augmentation.

Colab/Kaggle machines have only 2 CPU cores, so a normal torchvision DataLoader
becomes the bottleneck for ResNet-18 on CIFAR. Keeping the whole dataset on the
GPU (~150 MB as uint8) and augmenting there makes each epoch several times faster.
"""
import numpy as np
import torch
import torch.nn.functional as F
import torchvision

STATS = {
    "cifar10": ((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    "cifar100": ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
}
NUM_CLASSES = {"cifar10": 10, "cifar100": 100}


def load_cifar(name, root="./data", device="cuda", limit_train=None):
    """Returns (x_train, y_train, x_test, y_test) as tensors on `device`.

    Images are uint8 NCHW; normalisation happens per batch in `GpuLoader`.
    `limit_train` keeps only the first N training images (for quick debugging).
    """
    cls = {"cifar10": torchvision.datasets.CIFAR10,
           "cifar100": torchvision.datasets.CIFAR100}[name]
    tr = cls(root, train=True, download=True)
    te = cls(root, train=False, download=True)

    def to_t(ds, limit=None):
        x = torch.from_numpy(np.asarray(ds.data)).permute(0, 3, 1, 2).contiguous()
        y = torch.tensor(ds.targets, dtype=torch.long)
        if limit:
            x, y = x[:limit], y[:limit]
        return x.to(device), y.to(device)

    x_tr, y_tr = to_t(tr, limit_train)
    x_te, y_te = to_t(te, limit_train // 5 if limit_train else None)
    return x_tr, y_tr, x_te, y_te


class GpuLoader:
    """Minimal DataLoader replacement for tensors that already live on the GPU.

    Yields (x, y, w, idx): normalised float images, labels, per-sample loss
    weights (all ones unless given, e.g. CRAIG weights) and dataset indices.
    """

    def __init__(self, x, y, dataset, batch_size=256, train=True, weights=None,
                 indices=None, drop_last=False):
        self.x, self.y = x, y
        self.batch_size, self.train, self.drop_last = batch_size, train, drop_last
        dev = x.device
        self.indices = (torch.arange(len(x), device=dev) if indices is None
                        else torch.as_tensor(indices, device=dev, dtype=torch.long))
        self.weights = (torch.ones(len(self.indices), device=dev) if weights is None
                        else torch.as_tensor(weights, device=dev, dtype=torch.float))
        mean, std = STATS[dataset]
        self.mean = torch.tensor(mean, device=dev).view(1, 3, 1, 1) * 255
        self.std = torch.tensor(std, device=dev).view(1, 3, 1, 1) * 255

    def __len__(self):
        n = len(self.indices)
        return n // self.batch_size if self.drop_last else -(-n // self.batch_size)

    def _augment(self, x):
        # random crop 32x32 from 4-pixel zero padding, per sample
        b = x.shape[0]
        xp = F.pad(x, (4, 4, 4, 4))
        i = torch.randint(0, 9, (b,), device=x.device)
        j = torch.randint(0, 9, (b,), device=x.device)
        ar = torch.arange(32, device=x.device)
        rows = (i[:, None] + ar)[:, None, :, None]
        cols = (j[:, None] + ar)[:, None, None, :]
        x = xp[torch.arange(b, device=x.device)[:, None, None, None],
               torch.arange(3, device=x.device)[None, :, None, None], rows, cols]
        # random horizontal flip
        flip = torch.rand(b, device=x.device) < 0.5
        return torch.where(flip[:, None, None, None], x.flip(3), x)

    def __iter__(self):
        n = len(self.indices)
        order = torch.randperm(n, device=self.x.device) if self.train else torch.arange(n, device=self.x.device)
        for s in range(0, n, self.batch_size):
            pos = order[s:s + self.batch_size]
            if self.drop_last and len(pos) < self.batch_size:
                break
            idx = self.indices[pos]
            xb = self.x[idx].float()
            if self.train:
                xb = self._augment(xb)
            xb = (xb - self.mean) / self.std
            yield xb, self.y[idx], self.weights[pos], idx

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/finetune.py
"""Prune a dense checkpoint, select a training subset, fine-tune, evaluate.

Single run:
    python -m src.finetune --ckpt checkpoints/resnet18_cifar10.pt \
        --sparsity 0.9 --method craig --frac 0.1 --seed 0

Each run appends one JSON line to --results (default results/results.jsonl).
Many runs: see src/run_grid.py.
"""
import argparse
import platform
from pathlib import Path
import time
import uuid
import warnings

import torch
import torch.nn.functional as F

from .data import GpuLoader, NUM_CLASSES, load_cifar
from .models import resnet18
from .prune import global_magnitude_prune, measure_sparsity
from .selection import METHODS, NEEDS_DENSE, SelectionContext, select
from .audit import (SCHEMA_VERSION, completed_keys, experiment_identity, full_config,
                    git_metadata, identity_key, load_jsonl, run_key, utc_now)
from .utils import (append_jsonl, autocast, evaluate, get_device, per_sample_outputs,
                    set_seed, synchronize, timed)

ALL_METHODS = ["full"] + list(METHODS)


def build_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", required=True, help="dense checkpoint from src.train_dense")
    p.add_argument("--dataset", default="cifar10", choices=list(NUM_CLASSES))
    p.add_argument("--sparsity", type=float, default=0.9)
    p.add_argument("--method", default="random", choices=ALL_METHODS)
    p.add_argument("--frac", type=float, default=0.1, help="fraction of training data (ignored for 'full')")
    p.add_argument("--epochs", type=int, default=30, help="fine-tuning epochs over the subset")
    p.add_argument("--lr", type=float, default=0.01)
    p.add_argument("--batch_size", type=int, default=128)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--reselect_every", type=int, default=0,
                   help="re-run selection every R epochs with the current model (0 = select once)")
    p.add_argument("--no_weights", action="store_true", help="ignore CRAIG-style per-sample weights")
    p.add_argument("--damage_alpha", type=float, default=1.0, help="alpha for damage_fl")
    p.add_argument("--results", default="results/results.jsonl")
    p.add_argument("--tag", default="", help="free-form label stored with the result")
    p.add_argument("--data_root", default="./data")
    p.add_argument("--limit_train", type=int, default=None, help="debug: use first N images")
    return p


def load_dense(a, device):
    ck = torch.load(a.ckpt, map_location=device)
    model = resnet18(NUM_CLASSES[a.dataset]).to(device)
    model.load_state_dict(ck["state_dict"])
    return model


def weight_stats(w, probs, y, epoch):
    """Diagnostics for one selection: how skewed the per-sample training weights are,
    and how well the current model already fits the full training set."""
    w = w.float()
    top = w.sort(descending=True).values
    k1 = max(1, len(w) // 100)
    s = {"epoch": epoch, "w_max_over_mean": round(float(w.max() / w.mean()), 1),
         "w_top1pct_share": round(float(top[:k1].sum() / w.sum()), 3)}
    if probs is not None:
        s["train_acc_before_select"] = round(float((probs.argmax(1) == y).float().mean()) * 100, 2)
    return s


def make_subset(a, model, data, dense_loss, device, timings, epoch=0):
    """Return indices, weights and diagnostics; time scoring separately.

    `epoch` offsets the random generator so that re-selection (random method) draws a
    new subset each time; epoch 0 keeps the original seed, so one-shot runs are unchanged.
    """
    x_tr, y_tr = data[0], data[1]
    if a.method == "full":
        idx = torch.arange(len(y_tr), device=device)
        return idx, torch.ones(len(idx), device=device), {}
    loss = probs = feats = None
    if a.method != "random":
        with timed(timings, "pruned_scoring_time", device):
            loss, probs, feats = per_sample_outputs(model, x_tr, y_tr, a.dataset, device)
    ctx = SelectionContext(y=y_tr, num_classes=NUM_CLASSES[a.dataset], loss=loss, probs=probs,
                           feats=feats, dense_loss=dense_loss, damage_alpha=a.damage_alpha,
                           generator=torch.Generator().manual_seed(a.seed + 1000 * epoch))
    with timed(timings, "selection_time", device):
        idx, w = select(a.method, ctx, a.frac)
        if a.no_weights:
            w = torch.ones_like(w)
    return idx, w, weight_stats(w, probs, y_tr, epoch)


def run(a, data=None, verbose=True):
    """Journal one attempt. Incomplete attempts restart from the dense checkpoint.

    Dense losses are computed once per attempt, kept in memory for re-selection,
    and never reused across runs. Every attempt pays its own dense scoring cost.
    """
    start = time.perf_counter()
    started_at = utc_now()
    identity = experiment_identity(a)
    key = identity_key(identity)
    if key in completed_keys(load_jsonl(a.results)):
        if verbose:
            print(f"[{key}] already completed; skipping", flush=True)
        return None
    device = get_device()
    metadata = {
        "git": git_metadata(), "checkpoint_path": str(Path(a.ckpt).resolve()),
        "checkpoint_sha256": identity["checkpoint_sha256"],
        "source_sha256": identity["source_sha256"], "python_version": platform.python_version(),
        "pytorch_version": str(torch.__version__), "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "gpu_model": torch.cuda.get_device_name() if device == "cuda" else None,
        "device": device, "seed": a.seed,
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cuda_matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
        "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
        "dense_scoring_cache": "in-memory within this attempt only; no cross-run cache",
        "data_preloaded": data is not None,
        "wall_clock_scope": ("run entry through final evaluation/metadata; excludes final journal append; "
                             + ("shared data loading excluded" if data is not None else "standalone data loading included")),
        "timing_schema": "v2-disjoint-scoring-selection-training-evaluation",
        "unique_samples_seen_definition": "union of indices passed to training loader; scoring/evaluation excluded",
    }
    base = {**full_config(a), "schema_version": SCHEMA_VERSION, "key": key,
            "identity": identity, "config": full_config(a), "metadata": metadata,
            "attempt_id": uuid.uuid4().hex, "started_at": started_at}
    append_jsonl(a.results, {**base, "status": "started"})
    timings = {name: 0.0 for name in (
        "dense_scoring_time", "pruned_scoring_time", "selection_time", "train_time", "evaluation_time")}
    try:
        record = _run(a, data, verbose, device, timings, key)
        synchronize(device)
        record.update(base)
        record.update({k: round(v, 6) for k, v in timings.items()})
        record.update(status="completed", ended_at=utc_now(),
                      scoring_time=round(timings["dense_scoring_time"] + timings["pruned_scoring_time"], 6),
                      total_wall_time=round(time.perf_counter() - start, 6))
        # Legacy-compatible effective fraction; raw CLI settings are in config.
        record["frac"] = identity["frac"]
        append_jsonl(a.results, record)
        if verbose:
            print(f"  final {record['final_acc']:.2f}%; total {record['total_wall_time']:.2f}s -> {a.results}", flush=True)
        return record
    except BaseException as exc:
        failed = {**base, **{k: round(v, 6) for k, v in timings.items()},
                  "status": "interrupted" if isinstance(exc, (KeyboardInterrupt, SystemExit)) else "failed",
                  "ended_at": utc_now(), "error_type": type(exc).__name__, "error": str(exc),
                  "scoring_time": round(timings["dense_scoring_time"] + timings["pruned_scoring_time"], 6),
                  "total_wall_time": round(time.perf_counter() - start, 6)}
        try:
            append_jsonl(a.results, failed)
        except Exception as journal_error:
            warnings.warn(f"Could not append failure status: {journal_error}", RuntimeWarning)
        raise


def _run(a, data, verbose, device, timings, key):
    set_seed(a.seed)
    if data is None:
        data = load_cifar(a.dataset, a.data_root, device, a.limit_train)
    x_tr, y_tr, x_te, y_te = data

    model = load_dense(a, device)
    with timed(timings, "evaluation_time", device):
        dense_acc = evaluate(model, x_te, y_te, a.dataset, device)
    dense_loss = None
    if a.method in NEEDS_DENSE:
        with timed(timings, "dense_scoring_time", device):
            dense_loss, _, _ = per_sample_outputs(model, x_tr, y_tr, a.dataset, device)

    global_magnitude_prune(model, a.sparsity)
    sparsity = measure_sparsity(model)
    with timed(timings, "evaluation_time", device):
        pruned_acc = evaluate(model, x_te, y_te, a.dataset, device)

    idx, w, st = make_subset(a, model, data, dense_loss, device, timings)
    sel_stats = [st]
    seen = torch.zeros(len(y_tr), dtype=torch.bool, device=device)
    seen[idx] = True
    if verbose:
        print(f"[{key}] dense {dense_acc:.2f}% -> pruned {pruned_acc:.2f}% "
              f"(sparsity {sparsity:.3f}); subset {len(idx)} samples", flush=True)
        print(f"  weights: {st}", flush=True)

    loader = GpuLoader(x_tr, y_tr, a.dataset, a.batch_size, train=True, weights=w, indices=idx)
    opt = torch.optim.SGD(model.parameters(), lr=a.lr, momentum=0.9, weight_decay=a.wd)
    successful_updates = [0]
    def count_update(optimizer, args, kwargs):
        successful_updates[0] += 1
    opt.register_step_post_hook(count_update)
    total_steps = a.epochs * len(loader)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, total_steps)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

    curve, steps, samples_processed = [], 0, 0
    for ep in range(a.epochs):
        if a.reselect_every and ep > 0 and ep % a.reselect_every == 0:
            idx, w, st = make_subset(a, model, data, dense_loss, device, timings, epoch=ep)
            seen[idx] = True
            sel_stats.append(st)
            if verbose:
                print(f"  weights: {st}", flush=True)
            loader = GpuLoader(x_tr, y_tr, a.dataset, a.batch_size, train=True, weights=w, indices=idx)
        model.train()
        with timed(timings, "train_time", device):
            for xb, yb, wb, _ in loader:
                with autocast(device):
                    loss = (F.cross_entropy(model(xb), yb, reduction="none") * wb).mean()
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                if steps < total_steps:
                    sched.step()
                steps += 1
                samples_processed += len(yb)
        with timed(timings, "evaluation_time", device):
            acc = evaluate(model, x_te, y_te, a.dataset, device)
        curve.append(round(acc, 2))
        if verbose and ((ep + 1) % 5 == 0 or ep + 1 == a.epochs):
            print(f"  epoch {ep + 1}/{a.epochs}  test acc {acc:.2f}%", flush=True)

    record = {
        "frac": 1.0 if a.method == "full" else a.frac,
        "n_selected": int(len(idx)), "measured_sparsity": round(sparsity, 4),
        "dense_acc": round(dense_acc, 2), "pruned_acc": round(pruned_acc, 2),
        "final_acc": curve[-1], "best_acc": max(curve), "curve": curve,
        "sel_stats": sel_stats, "steps": steps, "sgd_steps": successful_updates[0],
        "skipped_optimizer_steps": steps - successful_updates[0],
        "samples_processed": samples_processed, "unique_samples_seen": int(seen.sum().item()),
        "device": torch.cuda.get_device_name() if device == "cuda" else device,
    }
    return record


if __name__ == "__main__":
    run(build_parser().parse_args())

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/models.py
"""ResNet-18 adapted to 32x32 CIFAR images (3x3 stem, no max-pool)."""
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNet(nn.Module):
    def __init__(self, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], 1)
        self.layer2 = self._make_layer(128, num_blocks[1], 2)
        self.layer3 = self._make_layer(256, num_blocks[2], 2)
        self.layer4 = self._make_layer(512, num_blocks[3], 2)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, n, stride):
        layers = []
        for s in [stride] + [1] * (n - 1):
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def features(self, x):
        """Penultimate (512-d) embedding, used by gradient-based selection."""
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        return F.adaptive_avg_pool2d(out, 1).flatten(1)

    def forward(self, x, return_features=False):
        h = self.features(x)
        logits = self.fc(h)
        return (logits, h) if return_features else logits


def resnet18(num_classes=10):
    return ResNet([2, 2, 2, 2], num_classes)

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/plot.py
"""Aggregate completed journals, keeping protocols and hardware separate."""
import argparse
import hashlib
import json
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from .audit import IDENTITY_FIELDS, canonical_json, identity_key, is_completed, load_jsonl

UNKNOWN = "<unknown>"
GROUP_FIELDS = [k for k in IDENTITY_FIELDS if k != "seed"] + [
    "checkpoint_id", "source_sha256", "protocol", "schema_version", "gpu",
    "runtime_id", "timing_schema", "data_preloaded", "wall_clock_scope",
]
CONDITION_FIELDS = [k for k in GROUP_FIELDS if k not in ("sparsity", "method", "frac")]
METRICS = ["pruned_acc", "dense_acc", "train_time", "selection_time", "scoring_time",
           "dense_scoring_time", "pruned_scoring_time", "evaluation_time", "total_wall_time",
           "steps", "sgd_steps", "unique_samples_seen", "samples_processed"]
COLORS = {"random": "#7f7f7f", "el2n": "#ff7f0e", "craig": "#1f77b4",
          "damage_topk": "#9467bd", "damage_fl": "#d62728"}


def normalized_record(record):
    config = record.get("config", record)
    identity = record.get("identity", {})
    meta = record.get("metadata", {})
    if record.get("schema_version") == 2 and record.get("key") != identity_key(identity):
        raise ValueError("v2 result has an inconsistent identity/key")
    row = {k: identity.get(k, config.get(k, UNKNOWN)) for k in IDENTITY_FIELDS}
    if row["method"] == "full":
        row["frac"] = 1.0
    row.update(
        checkpoint_id=identity.get("checkpoint_sha256", meta.get("checkpoint_sha256",
                                   config.get("ckpt", UNKNOWN))),
        source_sha256=identity.get("source_sha256", UNKNOWN),
        protocol=identity.get("protocol", "legacy-unknown-protocol"),
        schema_version=record.get("schema_version", 1),
        gpu=meta.get("gpu_model") or record.get("gpu") or record.get("device", UNKNOWN),
        runtime_id=canonical_json({k: meta.get(k, UNKNOWN) for k in (
            "pytorch_version", "cuda_version", "cudnn_version", "device",
            "deterministic_algorithms", "cudnn_benchmark", "cudnn_deterministic",
            "cuda_matmul_allow_tf32", "cudnn_allow_tf32")}),
        timing_schema=meta.get("timing_schema", "legacy-selection-includes-pruned-scoring"),
        data_preloaded=meta.get("data_preloaded", UNKNOWN),
        wall_clock_scope=meta.get("wall_clock_scope", UNKNOWN),
    )
    row = {k: "<none>" if v is None else v for k, v in row.items()}
    return {**row, **{k: record.get(k, float("nan")) for k in METRICS},
            "final_acc": record["final_acc"], "best_acc": record.get("best_acc", float("nan"))}


def aggregate_records(records, metric="final_acc"):
    rows, seen = [], {}
    for record in records:
        if not is_completed(record):
            continue
        row = normalized_record(record)
        if row["seed"] == UNKNOWN:
            raise ValueError("cannot aggregate a completed result without its seed")
        token = tuple(row[k] for k in GROUP_FIELDS) + (row["seed"],)
        encoded = canonical_json(record)
        if token in seen:
            if seen[token] != encoded:
                raise ValueError("conflicting completed results for the same configuration/seed; use distinct tags for intentional repeats")
            continue
        seen[token] = encoded
        rows.append(row)
    if not rows:
        raise ValueError("no completed results found")
    if any(r["schema_version"] == 1 for r in rows):
        warnings.warn("legacy results: missing configuration/provenance stays <unknown>; legacy timing is not v2 timing", RuntimeWarning)
    df = pd.DataFrame(rows)
    if df[metric].isna().any():
        raise ValueError(f"some completed results lack {metric}")
    aggregations = {"acc_mean": (metric, "mean"), "acc_std": (metric, "std"),
                    "n_seeds": ("seed", "nunique")}
    aggregations.update({k: (k, "mean") for k in METRICS})
    summary = df.groupby(GROUP_FIELDS, dropna=False, sort=False).agg(**aggregations).reset_index()
    front = ["dataset", "sparsity", "method", "frac", "epochs", "tag",
             "acc_mean", "acc_std", "n_seeds", "pruned_acc", "dense_acc",
             "train_time", "selection_time"]
    return summary[front + [c for c in summary.columns if c not in front]]


def figure_groups(summary):
    return list(summary.groupby(CONDITION_FIELDS, dropna=False, sort=False))


def write_outputs(summary, out, overwrite=False):
    out = Path(out)
    groups = figure_groups(summary)
    plans = []
    for condition, frame in groups:
        spec = dict(zip(CONDITION_FIELDS, condition))
        # Pandas may return numpy scalar keys, which need native JSON scalars.
        spec = {k: v.item() if hasattr(v, "item") else v for k, v in spec.items()}
        suffix = hashlib.sha256(canonical_json(spec).encode()).hexdigest()[:12]
        name = "acc_vs_frac.png" if len(groups) == 1 else f"acc_vs_frac_{suffix}.png"
        plans.append((out / name, spec, frame))
    targets = [out / "summary.csv", out / "figure_manifest.json"] + [p[0] for p in plans]
    if not overwrite and any(p.exists() for p in targets):
        raise FileExistsError("output already exists; choose a new --out directory (or explicitly --overwrite)")
    out.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out / "summary.csv", index=False)
    manifest = []
    for path, spec, frame in plans:
        sparsities = sorted(frame.sparsity.unique())
        fig, axes = plt.subplots(1, len(sparsities), figsize=(4.5 * len(sparsities), 3.8), squeeze=False)
        for ax, sp in zip(axes[0], sparsities):
            part = frame[frame.sparsity == sp]
            for method, sm in part.groupby("method", sort=False):
                if method == "full":
                    if len(sm) != 1:
                        raise ValueError("ambiguous full-data reference")
                    ax.axhline(sm.acc_mean.iloc[0], ls="--", c="k", label="full data (same configuration)")
                else:
                    sm = sm.sort_values("frac")
                    ax.errorbar(sm.frac * 100, sm.acc_mean, yerr=sm.acc_std.fillna(0),
                                marker="o", capsize=3, label=method, color=COLORS.get(method))
            ax.set_title(f"sparsity {sp:.0%}")
            ax.set_xlabel("fine-tuning data (%)")
            ax.set_ylabel("test accuracy (%)")
            ax.grid(alpha=.3)
            ax.legend(fontsize=7)
        fig.suptitle(f"epochs={spec['epochs']}, rs={spec['reselect_every']}, uniform={spec['no_weights']}, GPU={spec['gpu']}", fontsize=9)
        fig.tight_layout()
        fig.savefig(path, dpi=200)
        plt.close(fig)
        manifest.append({"file": path.name, "configuration": spec})
    (out / "figure_manifest.json").write_text(json.dumps(manifest, indent=2))
    return targets


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--results", nargs="+", default=["results/results.jsonl"])
    p.add_argument("--out", default="results/audit_figures")
    p.add_argument("--metric", default="final_acc", choices=["final_acc", "best_acc"])
    p.add_argument("--overwrite", action="store_true", help="explicitly replace generated outputs")
    a = p.parse_args()
    summary = aggregate_records([r for f in a.results for r in load_jsonl(f)], a.metric)
    for path in write_outputs(summary, a.out, a.overwrite):
        print(path)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/prune.py
"""Global unstructured magnitude pruning via torch.nn.utils.prune.

The pruning masks stay attached during fine-tuning (weight = weight_orig * mask),
so pruned weights remain exactly zero while the rest are updated.
"""
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune


def prunable_params(model, include_fc=True):
    params = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) or (include_fc and isinstance(m, nn.Linear)):
            params.append((m, "weight"))
    return params


def global_magnitude_prune(model, sparsity, include_fc=True):
    """Zero out the `sparsity` fraction of conv/linear weights with the smallest |w|."""
    if sparsity <= 0:
        return model
    prune.global_unstructured(prunable_params(model, include_fc),
                              pruning_method=prune.L1Unstructured, amount=sparsity)
    return model


@torch.no_grad()
def measure_sparsity(model, include_fc=True):
    zeros = total = 0
    for m, name in prunable_params(model, include_fc):
        w = getattr(m, name)
        zeros += (w == 0).sum().item()
        total += w.numel()
    return zeros / max(total, 1)


def make_permanent(model):
    """Fold masks into weights (call before saving a final pruned checkpoint)."""
    for m, name in prunable_params(model):
        if prune.is_pruned(m):
            prune.remove(m, name)
    return model

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/run_grid.py
"""Run a grid of (sparsity x method x frac x seed) experiments, resumably.

    python -m src.run_grid --ckpt /content/drive/MyDrive/cs260d/resnet18_cifar10.pt \
        --results /content/drive/MyDrive/cs260d/results.jsonl \
        --sparsities 0.9 --methods random craig el2n damage_fl --fracs 0.05 0.1 0.2 --seeds 0 1 2

* Only completed, verified v2 identities in --results are skipped. After a Colab
  disconnect, re-run the same command/commit; incomplete runs restart from dense.
* --shard i --num_shards 3 splits the grid into three independent shards:
  each person runs a different shard into their own results file. Never use
  concurrent writers in different Colab runtimes for the same Drive journal.
* 'full' (100% data) is run once per sparsity/seed regardless of --fracs.
* The dataset is loaded onto the GPU once and reused by every run.
"""
import argparse
import itertools

from .data import load_cifar
from .finetune import ALL_METHODS, build_parser, run
from .utils import get_device, load_jsonl
from .audit import completed_keys, experiment_identity, file_sha256, identity_key, source_sha256


def main():
    # allow_abbrev=False so finetune flags like --seed are not mistaken for --seeds
    g = argparse.ArgumentParser(allow_abbrev=False)
    g.add_argument("--sparsities", type=float, nargs="+", default=[0.9])
    g.add_argument("--methods", nargs="+", default=["full", "random", "craig", "el2n", "damage_fl"],
                   choices=ALL_METHODS)
    g.add_argument("--fracs", type=float, nargs="+", default=[0.05, 0.1, 0.2])
    g.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    g.add_argument("--shard", type=int, default=0)
    g.add_argument("--num_shards", type=int, default=1)
    g.add_argument("--dry_run", action="store_true", help="only list the runs")
    grid_args, rest = g.parse_known_args()
    base = build_parser().parse_args(rest + ["--method", "random"])   # shared settings

    configs = []
    for sp, m, seed in itertools.product(grid_args.sparsities, grid_args.methods, grid_args.seeds):
        for fr in ([1.0] if m == "full" else grid_args.fracs):
            a = argparse.Namespace(**vars(base))
            a.sparsity, a.method, a.frac, a.seed = sp, m, fr, seed
            configs.append(a)
    if grid_args.num_shards < 1 or not 0 <= grid_args.shard < grid_args.num_shards:
        g.error("require num_shards >= 1 and 0 <= shard < num_shards")
    configs = configs[grid_args.shard::grid_args.num_shards]

    records = load_jsonl(base.results)
    done = completed_keys(records)
    checkpoint_hash, source_hash = file_sha256(base.ckpt), source_sha256()
    keyed = [(a, identity_key(experiment_identity(a, checkpoint_hash, source_hash))) for a in configs]
    todo = [(a, key) for a, key in keyed if key not in done]
    legacy = sum(r.get("schema_version") != 2 and "final_acc" in r for r in records)
    if legacy:
        print(f"{legacy} legacy completions retained; missing v2 provenance, so they do not skip new runs")
    print(f"{len(configs)} runs in this shard, {len(configs) - len(todo)} already done, {len(todo)} to go")
    if grid_args.dry_run:
        for a, key in todo:
            print(f"  {key} sp={a.sparsity} method={a.method} frac={a.frac} epochs={a.epochs} seed={a.seed} rs={a.reselect_every} uniform={a.no_weights}")
        return

    if not todo:
        return

    data = load_cifar(base.dataset, base.data_root, get_device(), base.limit_train)
    for i, (a, _) in enumerate(todo, 1):
        print(f"\n=== run {i}/{len(todo)} ===")
        run(a, data=data)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/selection/__init__.py
"""Subset selection methods for post-pruning fine-tuning.

Every method has the signature  fn(ctx, frac) -> (indices, weights)
  indices : LongTensor of selected training-set indices
  weights : FloatTensor of per-sample loss weights (mean 1), same length

`ctx` is a SelectionContext holding everything computed once before selection:
labels, and per-sample outputs of the pruned model (and of the dense model).
All methods are class-balanced: each class gets round(frac * n_class) samples.

To add a method: write fn(ctx, frac) in a new file and register it in METHODS.
"""
from dataclasses import dataclass
from typing import Optional

import torch

from .random_sel import select_random
from .score_based import select_el2n, select_damage_topk
from .craig import select_craig, select_damage_fl


@dataclass
class SelectionContext:
    y: torch.Tensor                 # labels [N]
    num_classes: int
    loss: torch.Tensor              # per-sample loss of the PRUNED model [N]
    probs: torch.Tensor             # softmax of the pruned model [N, C]
    feats: torch.Tensor             # penultimate features of the pruned model [N, D]
    dense_loss: Optional[torch.Tensor] = None  # per-sample loss of the DENSE model [N]
    damage_alpha: float = 1.0       # see craig.select_damage_fl
    generator: Optional[torch.Generator] = None


METHODS = {
    "random": select_random,
    "el2n": select_el2n,
    "craig": select_craig,
    "damage_topk": select_damage_topk,
    "damage_fl": select_damage_fl,
}

NEEDS_DENSE = {"damage_topk", "damage_fl"}


def select(method, ctx, frac):
    if method not in METHODS:
        raise ValueError(f"unknown method {method!r}; choose from {list(METHODS)}")
    idx, w = METHODS[method](ctx, frac)
    w = w.float()
    return idx, w * (len(w) / w.sum())   # normalise weights to mean 1

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/selection/craig.py
"""CRAIG (Mirzasoleiman et al., ICML 2020) and the damage-weighted variant.

CRAIG picks a subset S whose weighted gradient sum approximates the full gradient
by greedily maximising the facility-location function

    F(S) = sum_i  w_i * max_{j in S} sim(i, j),     sim(i, j) = d_max - ||g_i - g_j||

where g_i is a cheap proxy for sample i's gradient. Each selected j then gets a
training weight gamma_j = number of samples i whose closest element in S is j.

Gradient proxy: the exact gradient of the loss w.r.t. the last linear layer,
    g_i = (p_i - y_i) (outer) [h_i, 1]
with p_i the softmax output and h_i the penultimate features. Pairwise distances
are computed without materialising the (C x (D+1))-dim vectors, using
    <a(x)b, c(x)d> = <a, c> <b, d>.

Plain CRAIG uses w_i = 1. `select_damage_fl` (our method) uses
    w_i = alpha + damage_i / mean(damage)
so samples that pruning hurt the most count more in the coverage objective,
while alpha > 0 keeps every sample represented (alpha -> inf recovers CRAIG).
"""
import torch
import torch.nn.functional as F

from .random_sel import per_class_budget
from .score_based import damage_scores


def _grad_distances(err, h):
    """Pairwise ||g_i - g_j|| for g = err (outer) [h, 1]."""
    h1 = torch.cat([h, torch.ones(len(h), 1, device=h.device)], dim=1)
    ee = err @ err.T
    hh = h1 @ h1.T
    dot = ee * hh                              # <g_i, g_j>
    sq = dot.diagonal()
    d2 = sq[:, None] + sq[None, :] - 2 * dot
    return d2.clamp(min=0).sqrt()


def facility_location_greedy(sim, k, w=None):
    """Greedy maximisation of sum_i w_i max_{j in S} sim[i, j].

    sim: [n, n] non-negative similarities (row = sample to cover, col = candidate).
    Returns (selected column indices, cluster sizes gamma_j).
    """
    n = sim.shape[0]
    w = torch.ones(n, device=sim.device) if w is None else w
    cur = torch.zeros(n, device=sim.device)         # current coverage of each row
    selected = []
    available = torch.ones(n, dtype=torch.bool, device=sim.device)
    for _ in range(min(k, n)):
        gains = (w[:, None] * (sim - cur[:, None]).clamp(min=0)).sum(0)
        gains[~available] = -1
        j = int(gains.argmax())
        selected.append(j)
        available[j] = False
        cur = torch.maximum(cur, sim[:, j])
    sel = torch.tensor(selected, device=sim.device)
    assign = sim[:, sel].argmax(1)                   # nearest selected element
    gamma = torch.bincount(assign, minlength=len(sel)).float()
    return sel, gamma


def _fl_select(ctx, frac, point_weights=None):
    onehot = F.one_hot(ctx.y, ctx.num_classes).float()
    err = ctx.probs.float() - onehot
    chosen, weights = [], []
    for c in range(ctx.num_classes):
        cls_idx = (ctx.y == c).nonzero(as_tuple=True)[0]
        if len(cls_idx) == 0:
            continue
        k = per_class_budget(len(cls_idx), frac)
        dist = _grad_distances(err[cls_idx], ctx.feats[cls_idx].float())
        sim = dist.max() - dist
        w = None if point_weights is None else point_weights[cls_idx]
        sel, gamma = facility_location_greedy(sim, k, w)
        chosen.append(cls_idx[sel])
        weights.append(gamma.clamp(min=1))
    return torch.cat(chosen), torch.cat(weights)


def select_craig(ctx, frac):
    return _fl_select(ctx, frac)


def select_damage_fl(ctx, frac):
    d = damage_scores(ctx)
    w = ctx.damage_alpha + d / d.mean().clamp(min=1e-8)
    return _fl_select(ctx, frac, point_weights=w)

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/selection/random_sel.py
import torch


def per_class_budget(n_c, frac):
    return max(1, int(round(frac * n_c)))


def select_random(ctx, frac):
    """Class-balanced uniform random subset (the key baseline)."""
    chosen = []
    for c in range(ctx.num_classes):
        cls_idx = (ctx.y == c).nonzero(as_tuple=True)[0]
        if len(cls_idx) == 0:
            continue
        k = per_class_budget(len(cls_idx), frac)
        perm = torch.randperm(len(cls_idx), generator=ctx.generator, device="cpu")
        chosen.append(cls_idx[perm[:k].to(cls_idx.device)])
    idx = torch.cat(chosen)
    return idx, torch.ones(len(idx), device=idx.device)

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/selection/score_based.py
"""Score-based selection: keep the top-scoring samples within each class."""
import torch
import torch.nn.functional as F

from .random_sel import per_class_budget


def _topk_per_class(ctx, scores, frac):
    chosen = []
    for c in range(ctx.num_classes):
        cls_idx = (ctx.y == c).nonzero(as_tuple=True)[0]
        if len(cls_idx) == 0:
            continue
        k = per_class_budget(len(cls_idx), frac)
        top = scores[cls_idx].topk(min(k, len(cls_idx))).indices
        chosen.append(cls_idx[top])
    idx = torch.cat(chosen)
    return idx, torch.ones(len(idx), device=idx.device)


def el2n_scores(ctx):
    """EL2N = || softmax(f(x)) - onehot(y) ||_2, computed with the pruned model."""
    onehot = F.one_hot(ctx.y, ctx.num_classes).float()
    return (ctx.probs - onehot).norm(dim=1)


def damage_scores(ctx):
    """How much pruning hurt each sample: loss_pruned - loss_dense (clipped at 0)."""
    if ctx.dense_loss is None:
        raise ValueError("damage scores need the dense model's per-sample loss")
    return (ctx.loss - ctx.dense_loss).clamp(min=0)


def select_el2n(ctx, frac):
    """Hardest samples for the pruned model."""
    return _topk_per_class(ctx, el2n_scores(ctx), frac)


def select_damage_topk(ctx, frac):
    """Samples whose loss increased the most because of pruning (no diversity term)."""
    return _topk_per_class(ctx, damage_scores(ctx), frac)

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/train_dense.py
"""Train the dense (unpruned) ResNet-18 that every experiment starts from.

    python -m src.train_dense --dataset cifar10 --epochs 40 --out checkpoints/resnet18_cifar10.pt

~40 epochs reaches roughly 94-95% on CIFAR-10. Do this ONCE and save the
checkpoint to Google Drive; subsequent runs reuse the same file.
"""
import argparse
import os
import time

import torch
import torch.nn.functional as F

from .data import GpuLoader, NUM_CLASSES, load_cifar
from .models import resnet18
from .utils import autocast, evaluate, get_device, set_seed


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", default="cifar10", choices=list(NUM_CLASSES))
    p.add_argument("--epochs", type=int, default=40)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--data_root", default="./data")
    p.add_argument("--limit_train", type=int, default=None, help="debug: use first N images")
    p.add_argument("--out", required=True)
    a = p.parse_args()

    set_seed(a.seed)
    device = get_device()
    x_tr, y_tr, x_te, y_te = load_cifar(a.dataset, a.data_root, device, a.limit_train)
    model = resnet18(NUM_CLASSES[a.dataset]).to(device)
    loader = GpuLoader(x_tr, y_tr, a.dataset, a.batch_size, train=True, drop_last=True)
    opt = torch.optim.SGD(model.parameters(), lr=a.lr, momentum=0.9, nesterov=True, weight_decay=a.wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, a.lr, epochs=a.epochs,
                                                steps_per_epoch=len(loader), pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

    for ep in range(a.epochs):
        model.train()
        t0 = time.time()
        for xb, yb, _, _ in loader:
            with autocast(device):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        acc = evaluate(model, x_te, y_te, a.dataset, device)
        print(f"epoch {ep + 1}/{a.epochs}  loss {loss.item():.3f}  test acc {acc:.2f}%  ({time.time() - t0:.1f}s)",
              flush=True)

    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "dataset": a.dataset, "test_acc": acc}, a.out)
    print(f"saved {a.out}  (test acc {acc:.2f}%)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/cs260d-pruning-coreset/src/utils.py
import random
import time
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn.functional as F

from .data import GpuLoader
from .audit import append_jsonl, load_jsonl


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def autocast(device):
    """fp16 autocast on CUDA (works on T4/P100/A100); no-op elsewhere."""
    if device == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cpu", enabled=False)


@torch.no_grad()
def evaluate(model, x, y, dataset, device, batch_size=1000):
    model.eval()
    correct = 0
    for xb, yb, _, _ in GpuLoader(x, y, dataset, batch_size, train=False):
        with autocast(device):
            correct += (model(xb).argmax(1) == yb).sum().item()
    return 100.0 * correct / len(x)


@torch.no_grad()
def per_sample_outputs(model, x, y, dataset, device, batch_size=1000):
    """One pass over the full training set without augmentation.

    Returns per-sample loss, softmax probabilities and penultimate features,
    which the selection methods use as scores / gradient proxies.
    """
    model.eval()
    losses, probs, feats = [], [], []
    for xb, yb, _, _ in GpuLoader(x, y, dataset, batch_size, train=False):
        with autocast(device):
            logits, h = model(xb, return_features=True)
        logits = logits.float()
        losses.append(F.cross_entropy(logits, yb, reduction="none"))
        probs.append(logits.softmax(1))
        feats.append(h.float())
    return torch.cat(losses), torch.cat(probs), torch.cat(feats)


def synchronize(device):
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()


@contextmanager
def timed(timings, name, device):
    """Elapsed wall time with device work synchronized at both boundaries."""
    synchronize(device)
    start = time.perf_counter()
    try:
        yield
    finally:
        synchronize(device)
        timings[name] = timings.get(name, 0.0) + time.perf_counter() - start

## 3. Require the existing dense checkpoint (never train it automatically)

In [ ]:
import os
assert os.path.isfile(CKPT), f'Existing checkpoint not found; check the path: {CKPT}'
from src.audit import file_sha256
print('checkpoint:', CKPT, 'SHA256:', file_sha256(CKPT))

## 4. Smoke test: one run (90% sparsity, CRAIG, 10% data, 5 epochs)

In [ ]:
!python -m src.finetune --ckpt "{CKPT}" --data_root "{DATA}" --sparsity 0.9 --method craig \
    --frac 0.1 --epochs 5 --results "{WORK}/results_audit_smoke_{NAME}.jsonl"

## 5. Main grid

Large historical grids below are disabled by default. Use COLAB_RUN_GUIDE.md for a small audit grid.
After disconnect, rerun setup with the same commit, checkpoint, flags and journal path.
Only completed v2 runs are skipped; interrupted runs restart from the dense checkpoint.
Use a separate journal for each runtime; do not write concurrently to one file.
Split work with SHARD/NUM_SHARDS only when intentionally running a larger grid.

In [ ]:
if globals().get('RUN_LARGE_GRIDS', False):
    SHARD, NUM_SHARDS = 0, 1
    !python -m src.run_grid --ckpt "{CKPT}" --data_root "{DATA}" --results "{RESULTS}" \
        --sparsities 0.9 0.95 0.98 --methods full random craig el2n damage_topk damage_fl \
        --fracs 0.05 0.1 0.2 --seeds 0 1 2 --epochs 30 \
        --shard {SHARD} --num_shards {NUM_SHARDS}
else:
    print('Skipped historical grid; see COLAB_RUN_GUIDE.md for small audit runs.')


## 6. Extra: re-select the subset every 5 epochs

Same 30-epoch budget, but the subset is re-chosen with the current model every 5 epochs.
Random is re-drawn too. Cumulative distinct data can differ across methods;
completed runs record the exact training union as unique_samples_seen.
Versioned identities distinguish these runs from one-shot runs.

In [ ]:
if globals().get('RUN_LARGE_GRIDS', False):
    !python -m src.run_grid --ckpt "{CKPT}" --data_root "{DATA}" --results "{RESULTS}" \
        --sparsities 0.9 0.95 0.98 --methods random craig damage_fl \
        --fracs 0.05 0.1 0.2 --seeds 0 1 2 --epochs 30 --reselect_every 5
else:
    print('Skipped historical grid; see COLAB_RUN_GUIDE.md for small audit runs.')


## 7. Diagnostic: how skewed are CRAIG weights when re-selecting?

Re-selected CRAIG / damage_fl runs sometimes collapse right after a re-selection. Each run now
prints `weights: {...}` at every selection: max weight / mean weight, the share of total weight
held by the top 1% of selected samples, and the model's training accuracy before selecting.
Four cases: two that collapsed, one stable control (5% data), one 98%-sparsity case.
Results go to a separate file so the main results are untouched.

In [ ]:
if globals().get('RUN_LARGE_GRIDS', False):
    for sp, m, fr, sd in [(0.9, "craig", 0.2, 2), (0.9, "craig", 0.05, 0), (0.9, "damage_fl", 0.2, 0), (0.98, "craig", 0.2, 0)]:
        !python -m src.finetune --ckpt "{CKPT}" --data_root "{DATA}" --results "{WORK}/results_audit_diag_{NAME}.jsonl" --sparsity {sp} --method {m} --frac {fr} --seed {sd} --epochs 30 --reselect_every 5 --tag diag
else:
    print('Skipped historical grid; see COLAB_RUN_GUIDE.md for small audit runs.')


## 8. Equal-compute baseline: full data, fewer epochs

A 20% subset for 30 epochs is about 2370 SGD steps; full data for 6 epochs is about 2346.
A 10% subset for 30 epochs (about 1200 steps) matches full data for 3 epochs (about 1170).
These runs answer whether selecting a subset beats simply training on all data for fewer epochs
at the same compute. Versioned identities distinguish epochs; this matches training steps approximately, not total wall time.

In [ ]:
if globals().get('RUN_LARGE_GRIDS', False):
    !python -m src.run_grid --ckpt "{CKPT}" --data_root "{DATA}" --results "{RESULTS}" \
        --sparsities 0.9 0.95 0.98 --methods full --seeds 0 1 2 --epochs 6
    !python -m src.run_grid --ckpt "{CKPT}" --data_root "{DATA}" --results "{RESULTS}" \
        --sparsities 0.9 0.95 0.98 --methods full --seeds 0 1 2 --epochs 3
else:
    print('Skipped historical grid; see COLAB_RUN_GUIDE.md for small audit runs.')


## 9. Plot only the selected audit journal into a fresh output directory

In [ ]:
import datetime
OUT = f"{WORK}/audit_figures_{NAME}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
if os.path.isfile(RESULTS):
    !python -m src.plot --results "{RESULTS}" --out "{OUT}"
    print('See summary.csv and figure_manifest.json in', OUT)
else:
    print('No audit grid journal yet; choose a small grid from COLAB_RUN_GUIDE.md.')